In [ ]:
!pip install -q ultralytics torch torchvision opencv-python matplotlib seaborn numpy pillow


In [ ]:
import os
import sys
import time
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator, colors


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active Device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1073741824:.2f} GB")
    torch.backends.cudnn.benchmark = True


In [ ]:
class AdvancedYOLOInferencePipeline:
    def __init__(self, model_variant="yolov8m.pt", task="detect"):
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.model = YOLO(model_variant)
        self.model.to(self.device)
        self.task = task
        self.classes = self.model.names

    def predict(self, source, conf=0.35, iou=0.5, agnostic_nms=False, max_det=300):
        start_time = time.perf_counter()
        results = self.model.predict(
            source=source,
            conf=conf,
            iou=iou,
            agnostic_nms=agnostic_nms,
            max_det=max_det,
            device=self.device,
            verbose=False
        )
        elapsed_ms = (time.perf_counter() - start_time) * 1000.0
        return results, elapsed_ms

    def parse_detections(self, result):
        boxes = result.boxes
        data = []
        if boxes is not None and len(boxes) > 0:
            for box in boxes:
                xyxy = box.xyxy[0].tolist()
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])
                cls_name = self.classes.get(cls_id, str(cls_id))
                data.append({
                    "bbox": xyxy,
                    "confidence": round(conf, 4),
                    "class_id": cls_id,
                    "class_name": cls_name,
                    "area": (xyxy[2] - xyxy[0]) * (xyxy[3] - xyxy[1])
                })
        return data

    def visualize_detections(self, result, title="Inference Output", figsize=(12, 8)):
        plotted_bgr = result.plot()
        plotted_rgb = cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(plotted_rgb)
        ax.axis("off")
        ax.set_title(title, fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()
        return plotted_rgb


In [ ]:
pipeline_detect = AdvancedYOLOInferencePipeline(model_variant="yolov8m.pt", task="detect")
sample_url = "https://ultralytics.com/images/bus.jpg"
results, latency = pipeline_detect.predict(sample_url, conf=0.4, iou=0.5)
detections = pipeline_detect.parse_detections(results[0])
print(f"Inference Latency: {latency:.2f} ms")
print(f"Total Objects Detected: {len(detections)}")
for det in detections[:5]:
    print(det)
pipeline_detect.visualize_detections(results[0], title=f"YOLOv8m Detection - Latency: {latency:.1f}ms")


In [ ]:
class YOLOSegmentationPipeline:
    def __init__(self, model_variant="yolov8n-seg.pt"):
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.model = YOLO(model_variant)
        self.model.to(self.device)

    def segment(self, source, conf=0.35, retina_masks=True):
        results = self.model.predict(
            source=source,
            conf=conf,
            retina_masks=retina_masks,
            device=self.device,
            verbose=False
        )
        return results

    def extract_masks_and_visualize(self, result, figsize=(12, 8)):
        plotted_bgr = result.plot()
        plotted_rgb = cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(plotted_rgb)
        ax.axis("off")
        ax.set_title("Instance Segmentation Mask Overlay", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()
        if result.masks is not None:
            print(f"Masks Detected: {len(result.masks.data)}")
            print(f"Mask Tensor Shape: {result.masks.data.shape}")


In [ ]:
pipeline_seg = YOLOSegmentationPipeline(model_variant="yolov8n-seg.pt")
seg_results = pipeline_seg.segment("https://ultralytics.com/images/zidane.jpg", conf=0.4)
pipeline_seg.extract_masks_and_visualize(seg_results[0])


In [ ]:
class ModelTelemetryBenchmark:
    def __init__(self, models=["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]):
        self.models = models
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    def run_benchmark(self, input_shape=(1, 3, 640, 640), warmups=5, iterations=20):
        telemetry = {}
        dummy_tensor = torch.randn(input_shape).to(self.device)
        for model_name in self.models:
            m = YOLO(model_name).model.to(self.device).eval()
            for _ in range(warmups):
                with torch.no_grad():
                    _ = m(dummy_tensor)
            if self.device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(iterations):
                with torch.no_grad():
                    _ = m(dummy_tensor)
            if self.device.type == "cuda":
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            mean_latency_ms = ((t1 - t0) / iterations) * 1000.0
            fps = 1000.0 / mean_latency_ms
            params = sum(p.numel() for p in m.parameters() if p.requires_grad)
            telemetry[model_name] = {
                "latency_ms": round(mean_latency_ms, 2),
                "fps": round(fps, 1),
                "parameters": params
            }
        return telemetry

    def plot_telemetry(self, telemetry):
        names = list(telemetry.keys())
        latencies = [telemetry[k]["latency_ms"] for k in names]
        fps_vals = [telemetry[k]["fps"] for k in names]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        sns.barplot(x=names, y=latencies, ax=ax1, palette="mako")
        ax1.set_title("Mean Inference Latency (ms)", fontweight="bold")
        ax1.set_ylabel("ms (lower is better)")
        for i, v in enumerate(latencies):
            ax1.text(i, v + 0.5, f"{v}ms", ha="center", fontweight="bold")
        sns.barplot(x=names, y=fps_vals, ax=ax2, palette="viridis")
        ax2.set_title("Throughput (Frames Per Second)", fontweight="bold")
        ax2.set_ylabel("FPS (higher is better)")
        for i, v in enumerate(fps_vals):
            ax2.text(i, v + 1.0, f"{v}", ha="center", fontweight="bold")
        plt.tight_layout()
        plt.show()


In [ ]:
benchmark = ModelTelemetryBenchmark(models=["yolov8n.pt", "yolov8s.pt"])
telemetry = benchmark.run_benchmark(iterations=10)
for model, stats in telemetry.items():
    print(f"{model}: {stats}")
benchmark.plot_telemetry(telemetry)


In [ ]:
class ModelExportOptimizer:
    @staticmethod
    def export_onnx(model_variant="yolov8n.pt", imgsz=(640, 640), dynamic=True, opset=17):
        model = YOLO(model_variant)
        export_path = model.export(format="onnx", imgsz=imgsz, dynamic=dynamic, opset=opset)
        return export_path

    @staticmethod
    def export_torchscript(model_variant="yolov8n.pt", imgsz=(640, 640)):
        model = YOLO(model_variant)
        export_path = model.export(format="torchscript", imgsz=imgsz)
        return export_path
